In [ ]:
#----------------------------------------
# Step 1 — install sb3
#----------------------------------------

# Run this once if packages are missing
#%pip install "stable-baselines3[extra]" gymnasium torch --quiet

#%pip install "numpy<2" --force-reinstall

In [1]:
#----------------------------------------
# Step 1.2 — import libraries
#----------------------------------------

import numpy as np # is used to handle numbers and arrays.
import stable_baselines3

import matplotlib
import gymnasium as gym #is used to create a proper RL environment.
from gymnasium import spaces

from stable_baselines3 import DDPG
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
import os

print("NumPy:", np.__version__)
print("Stable-Baselines3 imported successfully")

NumPy: 1.26.4
Stable-Baselines3 imported successfully


In [2]:
#----------------------------------------
# Step 2 - DDPG wrapper
#----------------------------------------

%run Maze_1_ML.ipynb

class ContinuousMazeWrapper(gym.Env):
    def __init__(self):
        super().__init__()

        self.env = MazeEnv()

        # Same observation space as your original MazeEnv
        self.observation_space = self.env.observation_space

        # DDPG needs a continuous action space
        # We use one continuous value between -1 and 1
        self.action_space = spaces.Box(
            low=np.array([-1.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
            dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        obs, info = self.env.reset(seed=seed, options=options)
        return obs, info

    def step(self, action):
        # Convert continuous DDPG action into one of 4 maze actions
        a = float(np.asarray(action).reshape(-1)[0])

        if a < -0.5:
            discrete_action = 0   # up
        elif a < 0.0:
            discrete_action = 1   # down
        elif a < 0.5:
            discrete_action = 2   # left
        else:
            discrete_action = 3   # right

        obs, reward, terminated, truncated, info = self.env.step(discrete_action)

        info["continuous_action"] = a
        info["discrete_action"] = discrete_action

        return obs, reward, terminated, truncated, info

    def render(self):
        self.env.render()

    def close(self):
        pass
        
        
        
print("Done!")


Done
Done!


In [3]:
#----------------------------------------
# Step 3 - Test the environment
#----------------------------------------

# Create one instance of the wrapped maze environment.
# This is the environment that DDPG will later use.
env = ContinuousMazeWrapper()

# Reset the environment to the starting position.
# obs is the first observation the agent receives.
# info is extra information from the environment.
obs, info = env.reset()

# Print the starting observation.
# In your case, this is the normalized player position.
print("Initial observation:", obs)

# Run 10 random test steps.
# This is not training yet.
# It only checks whether the environment works correctly.
for i in range(10):

    # Take a random continuous action from the action space.
    # Example: [-0.73], [0.12], [0.88]
    action = env.action_space.sample()

    # Send the action into the environment.
    # The wrapper converts the continuous action into a discrete maze action.
    # Then the maze moves the player if the move is valid.
    obs, reward, terminated, truncated, info = env.step(action)

    # Print what happened during this step.
    print(
        f"Step {i+1}: "
        # The original continuous action produced by the random sampler.
        f"continuous={info['continuous_action']:.2f}, "

        # The discrete action after conversion.
        # 0 = up, 1 = down, 2 = left, 3 = right.
        f"discrete={info['discrete_action']}, "

        # The new observation after the move.
        # This is the player's normalized position.
        f"obs={obs}, reward={reward}, "

        # done becomes True if the episode ended.
        # This happens if the agent reaches the exit or hits the max step limit.
        f"done={terminated or truncated}"
    )

    # If the episode ended, reset the environment so testing can continue.
    if terminated or truncated:
        obs, info = env.reset()

Initial observation: [0. 0.]
Step 1: continuous=0.95, discrete=3, obs=[0.         0.06666667], reward=0.9, done=False
Step 2: continuous=0.34, discrete=2, obs=[0. 0.], reward=-1.1, done=False
Step 3: continuous=-0.95, discrete=0, obs=[0. 0.], reward=-2.1, done=False
Step 4: continuous=-0.74, discrete=0, obs=[0. 0.], reward=-2.1, done=False
Step 5: continuous=0.27, discrete=2, obs=[0. 0.], reward=-2.1, done=False
Step 6: continuous=0.23, discrete=2, obs=[0. 0.], reward=-2.1, done=False
Step 7: continuous=-0.24, discrete=1, obs=[0.06666667 0.        ], reward=0.9, done=False
Step 8: continuous=0.92, discrete=3, obs=[0.06666667 0.        ], reward=-2.1, done=False
Step 9: continuous=-0.88, discrete=0, obs=[0. 0.], reward=-1.1, done=False
Step 10: continuous=0.30, discrete=2, obs=[0. 0.], reward=-2.1, done=False


In [8]:
#----------------------------------------
# Step 4 — Train DDPG
#----------------------------------------

# Create the wrapped maze environment.
# This is the environment that gives DDPG continuous actions.
env = ContinuousMazeWrapper()

# Check whether the environment follows the Gymnasium rules correctly.
# If something is wrong with reset(), step(), observation_space, or action_space,
# this function will warn us.
check_env(env, warn=True)

# Wrap the environment with Monitor.
# Monitor records training statistics such as episode reward and episode length.
env = Monitor(env)

# Create the DDPG model.
# "MlpPolicy" means the model uses a normal neural network.
model = DDPG(
    "MlpPolicy",          # Neural network policy type
    env,                  # Environment the model learns from
    verbose=1,            # Show training progress in the notebook
    learning_rate=1e-3,   # How strongly the model updates during learning
    buffer_size=50_000,   # How many past experiences are stored
    batch_size=64,        # How many experiences are used per training update
    gamma=0.99,           # How much the model values future rewards
)

# Train the model.
# The model will interact with the maze for 20,000 steps.
model.learn(total_timesteps=20_000)

# Create a folder called "models" if it does not already exist.
os.makedirs("models", exist_ok=True)

# Save the trained model inside the models folder.
model.save("models/ddpg_maze_quick")

# Print confirmation when the model has been saved.
print("Model saved.")

Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -140     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 61       |
|    time_elapsed    | 64       |
|    total_timesteps | 4000     |
| train/             |          |
|    actor_loss      | -0.935   |
|    critic_loss     | 0.019    |
|    learning_rate   | 0.001    |
|    n_updates       | 3899     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1e+03    |
|    ep_rew_mean     | -115     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 77       |
|    time_elapsed    | 103      |
|    total_timesteps | 8000     |
| train/             |          |
|    actor_loss      | 1.45     |
|    critic_loss     | 0.00534  |
|    learning_rate   | 0.001  

In [7]:
#----------------------------------------
# Step 5 - Test the trained model
#----------------------------------------

# Import the DDPG algorithm so we can load the saved trained model.
from stable_baselines3 import DDPG

# Create the same wrapped maze environment used during training.
env = ContinuousMazeWrapper()

# Load the trained model from the models folder.
# The env is passed in so the model knows which environment it should interact with.
model = DDPG.load("models/ddpg_maze_quick", env=env)

# Define a function that runs one full episode of the maze.
# One episode means: start at the beginning and continue until the agent
# reaches the exit or the maximum number of steps is reached.
def run_episode(model, env, deterministic=True, max_steps=500, render=True):

    # Reset the environment to the starting position.
    # obs is the initial observation given to the model.
    obs, info = env.reset()

    # Store the total reward collected during the episode.
    total_reward = 0

    # Run the episode for at most max_steps.
    for step in range(max_steps):

        # Ask the trained model to choose an action based on the current observation.
        # deterministic=True means the model chooses its best-known action,
        # instead of adding randomness.
        action, _ = model.predict(obs, deterministic=deterministic)

        # Send the chosen action into the environment.
        # The wrapper converts the continuous DDPG action into a maze movement.
        obs, reward, terminated, truncated, info = env.step(action)

        # Add this step's reward to the total episode reward.
        total_reward += reward

        # If render=True, show the maze after each movement.
        if render:
            env.render()

        # Stop the episode if the agent reached the exit
        # or if the environment ended for another reason.
        if terminated or truncated:
            break

    # Return the final results of the episode.
    return total_reward, step + 1, terminated, truncated


# Run 3 test episodes to see how the trained model performs.
for episode in range(3):

    # Run one episode using the trained model.
    total_reward, steps, terminated, truncated = run_episode(
        model,
        env,
        deterministic=True,
        render=True
    )

    # Print the result of the episode.
    print(
        f"Episode {episode+1}: "
        f"reward={total_reward:.2f}, "
        f"steps={steps}, "
        f"terminated={terminated}, "
        f"truncated={truncated}"
    )

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Step: 1 | Position: (1, 0) | Distance to exit: 29
Step: 2 | Position: (2, 0) | Distance to exit: 28
Step: 3 | Position: (3, 0) | Distance to exit: 27
Step: 4 | Position: (4, 0) | Distance to exit: 26
Step: 5 | Position: (5, 0) | Distance to exit: 25
Step: 6 | Position: (6, 0) | Distance to exit: 24
Step: 7 | Position: (7, 0) | Distance to exit: 23
Step: 8 | Position: (8, 0) | Distance to exit: 22
Step: 9 | Position: (9, 0) | Distance to exit: 21
Step: 10 | Position: (10, 0) | Distance to exit: 20
Step: 11 | Position: (11, 0) | Distance to exit: 19
Step: 12 | Position: (12, 0) | Distance to exit: 18
Step: 13 | Position: (13, 0) | Distance to exit: 17
Step: 14 | Position: (12, 0) | Distance to exit: 18
Step: 15 | Position: (13, 0) | Distance to exit: 17
Step: 16 | Position: (12, 0) | Distance to exit: 18
Step: 17 | Position: (13, 0) | Distance to exit: 17
Step: 18 | Position: (12, 0) | Distance to exit: 18
Step:

------------This is the END of the model---------------

In [ ]:
from stable_baselines3 import DDPG
import time
from IPython.display import clear_output

# Create environment
env = ContinuousMazeWrapper()

# Load trained model
model = DDPG.load("models/ddpg_maze_quick", env=env)

# Reset environment
obs, info = env.reset()

# Watch one episode
for step in range(500):
    action, _ = model.predict(obs, deterministic=True)

    obs, reward, terminated, truncated, info = env.step(action)

    clear_output(wait=True)
    env.render()

    print("Step:", step + 1)
    print("Reward:", reward)
    print("Done:", terminated or truncated)

    time.sleep(0.1)

    if terminated or truncated:
        break

In [ ]:
import os
os.getcwd()

In [ ]:
#git status
#git add .
#git commit -m "Update files"
#git push